
# Stack Overflow Developer Survey 2015 - canonical (not yet encoded) version

**Purpose:** one year of a shared multi-year dataset. This notebook produces `canonical_2015.csv`: shared column names, original text answers kept as they are for single-answer questions, multi-answer questions stored as `;`-separated lists (the format 2016+ files use), and the language write-ins cleaned into canonical names with a count. **No one-hot encoding, no imputation, no scaling, no rare-level merging** - those are learned from the data and belong in the pooled pipeline, fitted on the training years only.

**Same rules as the 2011 notebook**, plus the new things this year's file needs (listed in Gate 2). Nothing is assumed until printed; row counts are logged at every drop; one action per cell; every mapping is shown as a table.

**Gate 1 differs from 2011-2014:** the 2015 file has two header rows (the first line holds group labels such as 'Select all that apply', the second holds the column names) and no option-label row, so the notebook reads `header=1` and takes each tick-box's label from its own cells. It still uses `keep_default_na=False`, so real answers such as `None` survive.


## Gate 1 - Load and normalize

### Cell 1.0 - Imports, file path, decision log
**Purpose:** set up the environment and find the file. The path tries `data/2015/results.csv` first, then `2015_results.csv` in the working directory.
**Decision made:** none. `decision_log` collects every transformation so the final log is built from what actually ran.

In [1]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 300)

CANDIDATES = [Path("data/2015/results.csv"), Path("2015_results.csv")]
PATH = next((p for p in CANDIDATES if p.exists()), None)
if PATH is None:
    raise FileNotFoundError("none of the candidate paths exist: " + ", ".join(str(p) for p in CANDIDATES))
print("using file:", PATH, "| size in bytes:", PATH.stat().st_size)
print("pandas version:", pd.__version__)

decision_log = []

def log(step, text):
    decision_log.append((step, text))
    print("[LOG]", step, "-", text)

using file: data\2015\results.csv | size in bytes: 28483976
pandas version: 3.1.0


### Cell 1.1 - Detect the file encoding
**Purpose:** `read_csv` defaults to UTF-8; check that the file really decodes as UTF-8 before relying on it.
**Decision made:** use UTF-8 if the whole file decodes, otherwise `cp1252`. The 2015 file decodes as UTF-8, and its odd characters (`æ`, `Œ`) are **inside the file's text**, not a decoding problem: reading it as `cp1252` would only turn `æ` into `Ã¦`. They are repaired as text in Gate 3.

In [2]:
raw_bytes = PATH.read_bytes()
non_ascii = sum(1 for b in raw_bytes if b > 127)
print("total bytes:", len(raw_bytes), "| non-ASCII bytes:", non_ascii)
try:
    raw_bytes.decode("utf-8")
    ENCODING = "utf-8"
    print("utf-8 decodes the whole file cleanly")
except UnicodeDecodeError as err:
    ENCODING = "cp1252"
    print("utf-8 fails at byte", err.start, "- surrounding bytes:", raw_bytes[max(0, err.start - 25):err.start + 10])
    print("falling back to cp1252")
log("1.1 encoding", "file read with encoding=" + ENCODING + " (" + str(non_ascii) + " non-ASCII byte(s) in the file)")

total bytes: 28483976 | non-ASCII bytes: 115556
utf-8 decodes the whole file cleanly
[LOG] 1.1 encoding - file read with encoding=utf-8 (115556 non-ASCII byte(s) in the file)


### Cell 1.2 - Load with `header=1` (two header rows), without pandas' automatic NA parsing
**Purpose:** load the CSV exactly as written. Unlike 2015-2014, the first line of this file is **not** the header: it holds group labels (`Select all that apply`, `Select up to 3`) over the multi-answer blocks, and the real column names are on the second line.
**Decision made:** `header=1`; every cell read as text (`dtype=str`); `keep_default_na=False, na_values=[]` so real answers such as `None` survive. Blank cells are converted in Cell 1.5.

In [3]:
df = pd.read_csv(PATH, header=1, encoding=ENCODING, dtype=str, keep_default_na=False, na_values=[])
print("shape after load (rows, columns):", df.shape)
print("first 4 column names:", list(df.columns[:4]))
assert df.columns[0] == "Country", "header row 1 is not the column-name row"
log("1.2 load", "loaded " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " columns, all as text, header=1, default NA parsing off")

shape after load (rows, columns): (26086, 222)
first 4 column names: ['Country', 'Age', 'Gender', 'Tabs or Spaces']
[LOG] 1.2 load - loaded 26086 rows x 222 columns, all as text, header=1, default NA parsing off


### Cell 1.3 - Cross-check the load with Python's `csv` module and read the group-label row
**Purpose:** confirm pandas did not drop, split or merge rows, check that no header cell is empty, and print the group labels of the first line so the multi-answer blocks can be checked against them in Gate 2.
**Decision made:** none.

In [4]:
with open(PATH, encoding=ENCODING, newline="") as fh:
    csv_rows = list(csv.reader(fh))
print("rows seen by csv module (2 header rows + data):", len(csv_rows))
print("distinct row lengths:", sorted(set(len(r) for r in csv_rows)))
assert len(csv_rows) - 2 == len(df), "pandas and csv module disagree on row count"
assert all(len(r) == df.shape[1] for r in csv_rows), "ragged rows found"

group_row = csv_rows[0]
blank_header_positions = [i for i, h in enumerate(csv_rows[1]) if h.strip() == ""]
print("empty cells in the column-name row:", len(blank_header_positions))
assert not blank_header_positions
print("group labels on the first line (column -> label):")
for i, v in enumerate(group_row):
    if v.strip():
        print("   col", i, "(" + csv_rows[1][i][:45] + ") ->", v)

rows seen by csv module (2 header rows + data): 26088
distinct row lengths: [222]
empty cells in the column-name row: 0
group labels on the first line (column -> label):
   col 8 (Current Lang & Tech: Android) -> Select all that apply
   col 51 (Future Lang & Tech: Android) -> Select all that apply
   col 94 (Training & Education: No formal training) -> Select all that apply
   col 114 (Most important aspect of new job opportunity:) -> Select up to 3
   col 133 (Most annoying about job search: Finding time) -> Select up to 3
   col 149 (Appealing message traits: Message is personal) -> Select up to 3
   col 157 (Most urgent info about job opportunity: Salar) -> Select all that apply
   col 170 (How can companies improve interview process: ) -> Select up to 3
   col 179 (Why try Stack Overflow Careers: No spam) -> Select all that apply
   col 191 (Source control used: Git) -> Select all that apply
   col 204 (Why use Stack Overflow: Help for job) -> Select all that apply
   col 214 (Why

### Cell 1.4 - Strip whitespace from column names
**Purpose:** remove leading/trailing spaces from headers.
**Decision made:** `df.columns = df.columns.astype(str).str.strip()`. The cell lists which names changed.

In [5]:
changed = [(i, repr(c)) for i, c in enumerate(df.columns) if c != c.strip()]
print("column names with leading/trailing whitespace:", changed)
df.columns = df.columns.astype(str).str.strip()
print("duplicate column names:", df.columns[df.columns.duplicated()].tolist())
log("1.4 strip", str(len(changed)) + " column name(s) had surrounding whitespace and were stripped")

column names with leading/trailing whitespace: []
duplicate column names: []
[LOG] 1.4 strip - 0 column name(s) had surrounding whitespace and were stripped


### Cell 1.5 - Normalize blank cells to NaN
**Purpose:** turn empty and whitespace-only cells into NaN, once.
**Decision made:** `df.replace(r"^\s*$", np.nan, regex=True)`. The cell counts empty and whitespace-only cells first, then checks that the NaN count rose by exactly that amount. Literal text such as `None` or `N/A` is **not** touched; it is listed so you can see it.

In [6]:
empty_exact = int((df == "").sum().sum())
ws_only = int(df.apply(lambda s: s.str.match(r"^\s+$")).sum().sum())
na_before = int(df.isna().sum().sum())
print("cells exactly empty:", empty_exact, "| whitespace-only (non-empty):", ws_only, "| NaN before:", na_before)

df = df.replace(r"^\s*$", np.nan, regex=True)

na_after = int(df.isna().sum().sum())
print("NaN after:", na_after, "| increase:", na_after - na_before)
assert na_after - na_before == empty_exact + ws_only, "NaN count did not move by the expected amount"
print("total cells:", df.size, "| share missing: %.1f%%" % (100 * na_after / df.size))

literals = ["None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"]
found = []
for i, col in enumerate(df.columns):
    n = int(df.iloc[:, i].isin(literals).sum())
    if n:
        found.append((i, col[:50], n))
print("literal NA-like text kept as real values (not converted):")
for item in found:
    print("   col", item[0], "|", item[1], "|", item[2], "cells")
log("1.5 blanks", str(empty_exact + ws_only) + " empty/whitespace cells converted to NaN; " + str(sum(f[2] for f in found)) + " literal 'None'-style cells kept as text")

cells exactly empty: 4507861 | whitespace-only (non-empty): 0 | NaN before: 0
NaN after: 4507861 | increase: 4507861
total cells: 5791092 | share missing: 77.8%
literal NA-like text kept as real values (not converted):
   col 0 | Country | 807 cells
   col 50 | Current Lang & Tech: Write-In | 2 cells
   col 93 | Future Lang & Tech: Write-In | 2 cells
   col 186 | How many hours programming as hobby per week? | 1365 cells
   col 189 | Preferred text editor: write-in | 2 cells
   col 203 | Prefered Source Control: write-in | 21 cells
[LOG] 1.5 blanks - 4507861 empty/whitespace cells converted to NaN; 2199 literal 'None'-style cells kept as text


### Cell 1.6 - Option labels (there is no option-label row in 2015) and respondent numbering
**Purpose:** in 2015-2014 row 0 of the data held the label of each tick-box. 2015 has no such row, but every tick-box cell contains its own label (all cells of a tick-box column are the same text), so the label is read from the cells.
**Decisions made:**
- A column with exactly one distinct value is a tick-box and its label is that value, with the file's text artifacts (`æ`, `Œæ` standing in for a space) cleaned the same way Gate 3 cleans the cells. **One label contains a semicolon** ("No formal training; I'm self-taught"), which would break the `;`-separated list format, so its `;` becomes a `,` (Gate 3 applies the same change to the cells).
- A column whose name ends in `write-in` is a free-text box: its label is set to `Other (please specify)`, which is how the template recognises free-text boxes.
- Every other column is a single-answer question: label `Response`.
- The respondents are numbered from 1 in file order (`df.index`), which is the `ResponseId`, the same convention as 2015-2014.

In [7]:
def _clean_label(v):
    return v.replace("\u0152\u00e6", " ").replace("\u00e6", " ").replace(";", ",").strip()

labels = []
for i, c in enumerate(df.columns):
    s = df.iloc[:, i].dropna()
    if c.lower().endswith("write-in"):
        labels.append("Other (please specify)")
    elif s.nunique() == 1:
        labels.append(_clean_label(s.iloc[0]))
    else:
        labels.append("Response")
option_labels = pd.Series(labels, index=df.columns)
df.index = range(1, len(df) + 1)
print("tick-box columns:", int((option_labels.values != "Response").sum() - sum(l == "Other (please specify)" for l in labels)),
      "| free-text boxes:", sum(l == "Other (please specify)" for l in labels),
      "| single-answer columns:", sum(l == "Response" for l in labels))
print("respondents numbered", df.index.min(), "to", df.index.max())
log("1.6 option labels", "no option-label row in 2015; labels read from the tick-box cells; respondents numbered 1.." + str(len(df)))

tick-box columns: 178 | free-text boxes: 6 | single-answer columns: 38
respondents numbered 1 to 26086
[LOG] 1.6 option labels - no option-label row in 2015; labels read from the tick-box cells; respondents numbered 1..26086


### Cell 1.7 - Fully empty rows
**Purpose:** count respondents with no answers at all. Nothing is dropped in Gate 1; Gate 4 proposes the criteria.
**Decision made:** none.

In [8]:
filled_per_row = df.notna().sum(axis=1)
fully_empty = int((filled_per_row == 0).sum())
print("data rows:", len(df))
print("fully empty rows:", fully_empty, "| dropped in Gate 1: 0")
print("filled cells per row - lowest values (value : number of rows):")
print(filled_per_row.value_counts().sort_index().head(8).to_string())
only_country = int(((filled_per_row == 1) & df.iloc[:, 0].notna()).sum())
print("rows whose ONLY filled cell is Country (col 0):", only_country, "- candidates for Gate 4, not dropped")
log("1.7 empty rows", str(fully_empty) + " fully empty rows found; 0 dropped; " + str(only_country) + " rows have only Country filled (Gate 4 decision)")

data rows: 26086
fully empty rows: 0 | dropped in Gate 1: 0
filled cells per row - lowest values (value : number of rows):
1      48
2     101
3      84
4    1061
5    2411
6      30
7      34
8      47
rows whose ONLY filled cell is Country (col 0): 48 - candidates for Gate 4, not dropped
[LOG] 1.7 empty rows - 0 fully empty rows found; 0 dropped; 48 rows have only Country filled (Gate 4 decision)


### Cell 1.8 - Indexed column list
**Purpose:** print every column with its index, name, label and how many rows have a value.
**Decision made:** none.

In [9]:
records = []
for i, col in enumerate(df.columns):
    n_filled = int(df.iloc[:, i].notna().sum())
    records.append({"idx": i, "header": col[:70], "label": option_labels.iloc[i][:34], "n_filled": n_filled,
                    "pct_missing": round(100 * (1 - n_filled / len(df)), 1)})
col_index = pd.DataFrame(records)
print(col_index.to_string(index=False))

 idx                                                                 header                              label  n_filled  pct_missing
   0                                                                Country                           Response     26086          0.0
   1                                                                    Age                           Response     25831          1.0
   2                                                                 Gender                           Response     25744          1.3
   3                                                         Tabs or Spaces                           Response     25807          1.1
   4                                      Years IT / Programming Experience                           Response     24827          4.8
   5                                                             Occupation                           Response     20851         20.1
   6                                               Desktop Ope

### Cell 1.9 - Gate 1 summary
**Purpose:** final shape and the log so far.
**Decision made:** none. Gate 2 follows.

In [10]:
print("final shape (rows, columns):", df.shape)
print("data rows in file:", len(csv_rows) - 2, "| rows in df:", len(df))
print()
print("decision log so far:")
for step, text in decision_log:
    print("  ", step, "|", text)

final shape (rows, columns): (26086, 222)
data rows in file: 26086 | rows in df: 26086

decision log so far:
   1.1 encoding | file read with encoding=utf-8 (115556 non-ASCII byte(s) in the file)
   1.2 load | loaded 26086 rows x 222 columns, all as text, header=1, default NA parsing off
   1.4 strip | 0 column name(s) had surrounding whitespace and were stripped
   1.5 blanks | 4507861 empty/whitespace cells converted to NaN; 2199 literal 'None'-style cells kept as text
   1.6 option labels | no option-label row in 2015; labels read from the tick-box cells; respondents numbered 1..26086
   1.7 empty rows | 0 fully empty rows found; 0 dropped; 48 rows have only Country filled (Gate 4 decision)


## Gate 2 - Freeze names, choices, junk rows

### Cell 2.0 - Column plan and `CONFIG`
**Purpose:** one table (`ITEMS`) says which source column(s) become which output column, in questionnaire order. Every source column is checked against a keyword from its question text, and the cell proves that **no source column is left unused** (nothing is dropped silently).
**Kinds:** `single` = copied as typed; `list` = option columns joined into a `;` list; `freetext` = free-text box kept as typed (marked in the header map so you can drop it before modelling); `lang` = language block (Gate 6); `salary` = band text + band midpoint.
**Decisions made (switchable in `CONFIG`):** the rows dropped are the same as in 2011 (junk rows, no salary, `Student / Unemployed`) **plus `Rather not say`**, a salary answer that did not exist in 2011: it carries no amount, like a blank.

In [11]:
import re
import math
import datetime

RAW = list(df.columns)
PARENT = {}
_p = None
for _i in range(len(RAW)):
    if _i not in blank_header_positions:
        _p = _i
    PARENT[_i] = _p

def label(i):
    return option_labels.iloc[i]

def full_header(i):
    return RAW[i] if i not in blank_header_positions else RAW[PARENT[i]] + " -> " + str(label(i))

# kind, canonical name, status, source column (or (first, last+1)), keyword that must appear, note
ITEMS = [
    ("id", "ResponseId", "proposed", None, None, "respondent number in the 2015 file, counted from 1 (the file has no respondent id)"),
    ("year", "survey_year", "proposed", None, None, "constant 2015"),
    ("single", "Country", "confirmed", 0, "country", "real country names (157 values)"),
    ("single", "Age", "confirmed", 1, "age", "text bands, kept as typed (includes 'Prefer not to disclose')"),
    ("single", "Gender", "proposed", 2, "gender", "Male / Female / Other / Prefer not to disclose; think about whether it belongs in a salary model"),
    ("single", "TabsOrSpaces", "proposed", 3, "tabs or spaces", "new in 2015"),
    ("single", "YearsCode", "confirmed", 4, "years it", "text repaired to '<2','2-5','6-10','11+' (your mapping; 'Less than 1 year' also -> '<2', an addition to it); question says IT/programming experience"),
    ("single", "DevType", "confirmed", 5, "occupation", "single answer; 24 values, list differs from 2014"),
    ("single", "OpSys", "confirmed", 6, "desktop operating system", "text as typed"),
    ("freetext", "OpSys_WriteIn", "proposed", 7, "write-in", "typed text, raw"),
    ("lang", "LanguageHaveWorkedWith", "confirmed", (8, 51), "current lang", "';' list of canonical names; tick-boxes + parsed write-ins; 2015 asks for languages AND technologies, so tools go to ToolsHaveWorkedWith"),
    ("list", "TechWantToWorkWith", "proposed", (51, 94), "future lang", "';' list of options ticked (languages and technologies together); the 2015 analogue of 2013/2014 TechExcitedAbout and of later LanguageWantToWorkWith; NOT language-cleaned; typed 'Other' is in TechWantToWorkWith_WriteIn"),
    ("freetext", "TechWantToWorkWith_WriteIn", "proposed", 93, "write-in", "typed text, raw"),
    ("list", "TrainingAndEducation", "proposed", (94, 105), "training", "select all that apply; the label \"No formal training; I'm self-taught\" had its ';' changed to ',' so the ';' list stays unambiguous"),
    ("salary", "CompTotal", "confirmed", 105, "compensation", "salary BAND text, kept as typed except your repairs ('Less than $20,000' -> '<$20,000', 'More than $160,000' -> '>$160,000'); 'Unemployed' -> '0' then dropped"),
    ("single", "EmploymentStatus", "proposed", 107, "employment status", "new in 2015 (includes 'I'm a student', 'Retired')"),
    ("single", "Industry", "proposed", 108, "industry", "20 values, list differs from 2014"),
    ("single", "JobSat", "confirmed", 109, "job satisfaction", "six text answers; 2015 wording, incl. 'Other (please specify)'; later years use a 0-10 score, harmonise later"),
    ("single", "PurchasingPower", "proposed", 110, "purchasing power", "a single answer in 2015 (a ';' list called PurchaseInvolvement in 2011-2014)"),
    ("single", "RemoteWork", "proposed", 111, "remote status", "levels differ from 2014 ('I rarely work remote' replaces 'Occasionally')"),
    ("single", "ChangedJobsLast12m", "proposed", 112, "changed jobs", "Yes / No / 'No (I'm a student)'"),
    ("single", "OpenToNewJob", "proposed", 113, "open to new job", "three text answers"),
    ("list", "JobAspectsMostImportant", "proposed", (114, 132), "most important aspect", "select up to 3"),
    ("single", "RemoteImportance", "proposed", 132, "important is remote", "five text answers"),
    ("list", "JobSearchAnnoyances", "proposed", (133, 140), "most annoying", "select up to 3"),
    ("single", "RecruiterContactFrequency", "proposed", 140, "contacted by recruiters", "five text answers"),
    ("single", "RecruiterContactPerception", "proposed", 141, "perception of recruiter contact", "text artifacts ('Œæ' for a space) repaired"),
    ("single", "ContactPref_Email", "proposed", 142, "contact form: email", "Great / Tolerate / Hate / I don't have an account; the 2014 ContactPref_* columns used a different scale"),
    ("single", "ContactPref_LinkedIn", "proposed", 143, "contact form: linkedin", "same scale"),
    ("single", "ContactPref_Xing", "proposed", 144, "contact form: xing", "same scale; new in 2015"),
    ("single", "ContactPref_Phone", "proposed", 145, "contact form: phone", "same scale"),
    ("single", "ContactPref_CareersMessage", "proposed", 146, "contact form: stack overflow careers", "same scale"),
    ("single", "ContactPref_Twitter", "proposed", 147, "contact form: twitter", "same scale"),
    ("single", "ContactPref_Facebook", "proposed", 148, "contact form: facebook", "same scale; new in 2015"),
    ("list", "RecruitEmailAppealingTraits", "proposed", (149, 157), "appealing message traits", "select all that apply (2014 asked these as extent ratings, RecruitEmail_*)"),
    ("list", "JobInfoMostUrgent", "proposed", (157, 165), "most urgent info", "select up to 3"),
    ("single", "CommunicateWith_Headhunter", "proposed", 165, "headhunter", "Great / Tolerate / Hate"),
    ("single", "CommunicateWith_InHouseRecruiter", "proposed", 166, "in-house recruiter", "Great / Tolerate / Hate"),
    ("single", "CommunicateWith_InHouseTechRecruiter", "proposed", 167, "in-house tech recruiter", "Great / Tolerate / Hate"),
    ("single", "CommunicateWith_Manager", "proposed", 168, "communicate with about a new job opportunity: manager", "Great / Tolerate / Hate"),
    ("single", "CommunicateWith_Developer", "proposed", 169, "communicate with about a new job opportunity: developer", "Great / Tolerate / Hate"),
    ("list", "InterviewImprovements", "proposed", (170, 179), "improve interview", "select up to 3; text artifacts repaired"),
    ("list", "WhyTryCareers", "proposed", (179, 185), "why try stack overflow careers", "select all that apply; text artifacts repaired"),
    ("single", "CaffeinatedBeveragesPerDay", "proposed", 185, "caffeinated", "text '0'..'9' and 'More than 10', kept as typed"),
    ("single", "HobbyProgrammingHours", "proposed", 186, "hobby", "text bands, kept as typed; includes a real answer 'None'"),
    ("single", "StackOverflowVisitFrequency", "proposed", 187, "land on or read", "text artifacts repaired ('Multipleætimes a day')"),
    ("single", "PreferredEditor", "proposed", 188, "text editor", "text as typed; 'write-in' answers are in PreferredEditor_WriteIn"),
    ("freetext", "PreferredEditor_WriteIn", "proposed", 189, "write-in", "typed text, raw"),
    ("single", "IDETheme", "proposed", 190, "ide theme", "Dark / Light / I don't use an IDE"),
    ("list", "SourceControlUsed", "proposed", (191, 202), "source control used", "select all that apply; 'Other' means the typed answer in SourceControlUsed_WriteIn"),
    ("freetext", "SourceControlUsed_WriteIn", "proposed", 201, "write-in", "typed text, raw"),
    ("single", "PreferredSourceControl", "proposed", 202, "source control", "text as typed"),
    ("freetext", "PreferredSourceControl_WriteIn", "proposed", 203, "write-in", "typed text, raw"),
    ("list", "WhyUseStackOverflow", "proposed", (204, 213), "why use stack overflow", "select all that apply; text artifacts repaired"),
    ("single", "StackOverflowAnswersHelpful", "proposed", 213, "helpful", "five text answers"),
    ("list", "WhyAnswer", "proposed", (214, 222), "why answer", "select all that apply"),
]

covered = set()
for kind, name, status, spec, expect, note in ITEMS:
    if spec is None:
        continue
    first = spec[0] if isinstance(spec, tuple) else spec
    text = (full_header(first) + " " + str(label(first))).lower()
    assert expect.lower() in text, (name, expect, text[:120])
    covered |= set(range(spec[0], spec[1])) if isinstance(spec, tuple) else {spec}
    print("ok  %-30s %-8s cols %-10s | %s" % (name, kind, str(spec), full_header(first)[:60]))
UNUSED = {}

for i, why in UNUSED.items():
    assert int(df[RAW[i]].notna().sum()) == 0, "column " + str(i) + " has answers, so it cannot be declared unused"
    print("unused on purpose (no respondent answered it): col", i, "|", full_header(i)[:70], "|", why)
    log("2.0 unused column", "col " + str(i) + " has 0 answers and is not carried into the output: " + why)
covered |= set(UNUSED)
REDUNDANT = {}
REDUNDANT = {106: "the file's own midpoint of the salary band; compared with the midpoint used here in Cell 4.3 and not carried, so there are not two columns saying the same thing"}
for i, why in REDUNDANT.items():
    print("not carried into the output (redundant, checked in Gate 4): col", i, "|", full_header(i)[:70], "|", why)
    log("2.0 redundant column", "col " + str(i) + " not carried into the output: " + why)
covered |= set(REDUNDANT)
uncovered = [i for i in range(len(RAW)) if i not in covered]
print("source columns used by no output column:", uncovered)
assert not uncovered, "a source column would be dropped silently"

CONFIG = {"drop_rather_not_say": True, "drop_student_unemployed": True, "top_band_midpoint": 170000}
print(CONFIG)

row_log = [("1.2 two header rows read; no option-label row in the data", len(csv_rows) - 2, len(df))]
def log_rows(step, before, after):
    row_log.append((step, before, after))
    print("[ROWS]", step, ":", before, "->", after, "(removed", before - after, ")")

ok  Country                        single   cols 0          | Country
ok  Age                            single   cols 1          | Age
ok  Gender                         single   cols 2          | Gender
ok  TabsOrSpaces                   single   cols 3          | Tabs or Spaces
ok  YearsCode                      single   cols 4          | Years IT / Programming Experience
ok  DevType                        single   cols 5          | Occupation
ok  OpSys                          single   cols 6          | Desktop Operating System
ok  OpSys_WriteIn                  freetext cols 7          | Desktop Operating System: write-in
ok  LanguageHaveWorkedWith         lang     cols (8, 51)    | Current Lang & Tech: Android
ok  TechWantToWorkWith             list     cols (51, 94)   | Future Lang & Tech: Android
ok  TechWantToWorkWith_WriteIn     freetext cols 93         | Future Lang & Tech: Write-In
ok  TrainingAndEducation           list     cols (94, 105)  | Training & Education: No formal

## Gate 3 - Excel date damage (repairs, before any row is dropped)

### Cell 3.1 - Repair damaged band labels and check every column against its expected levels
**Purpose:** Excel turned some band labels into dates or numbers when the file was saved (`2-5` became a date, `11+` became `11`). This cell lists every raw value of each affected column with its repair, applies it, and asserts that **every value left is one of the expected levels**. It runs before any rows are dropped, so the counts below are whole-file counts you can compare with your own checks.
**Decision made:** the repairs are the ones from your notes (dates and bare numbers back to the survey's band labels; `Just me!` -> `1`, `Don't have an account` -> `0`, reputation levels `1000` -> `1000+` and so on). Columns with an empty repair table only get the level check. Numeric Excel serials are decoded below so you can see which date each one is. Every other single-answer column keeps its text exactly as typed.

In [12]:
def _clean(v):
    return v.replace("\u0152\u00e6", " ").replace("\u00e6", " ").replace(";", ",").strip()

COMP_ORDER = ["0", "<$20,000", "$20,000 - $40,000", "$40,000 - $60,000", "$60,000 - $80,000", "$80,000 - $100,000",
              "$100,000 - $120,000", "$120,000 - $140,000", "$140,000 - $160,000", ">$160,000", "Rather not say"]
FIXES = {
    4: ({"Less than 1 year": "<2", "1 - 2 years": "<2", "2 - 5 years": "2-5", "6 - 10 years": "6-10", "11+ years": "11+"}, ["<2", "2-5", "6-10", "11+"]),
    105: ({"Unemployed": "0", "Less than $20,000": "<$20,000", "More than $160,000": ">$160,000"}, COMP_ORDER),
}
# text artifacts: in every column with at most 40 distinct values (tick-boxes and single answers), 'AE' / 'OEAE' stand in for a space
auto = {}
for _i in range(len(RAW)):
    _s = df[RAW[_i]].dropna()
    if _i not in FIXES and _s.nunique() <= 40:
        _m = {v: _clean(v) for v in _s.unique() if _clean(v) != v}
        if _m:
            auto[_i] = (_m, sorted(set(_clean(v) for v in _s.unique())))
print("columns with the text artifact (cleaned like your clean_survey_col, levels = what is observed after cleaning):", sorted(auto))
FIXES.update(auto)

df_raw = df.copy()   # untouched copy, used by the validation in Gate 8
repaired_cols = []
for idx, (mapping, order) in FIXES.items():
    col = RAW[idx]
    before = df[col].value_counts(dropna=False)
    print("---", idx, "|", full_header(idx)[:90])
    table = pd.DataFrame({"raw_value": ["(blank)" if pd.isna(v) else str(v) for v in before.index], "n_rows": before.values})
    table["repaired_to"] = [mapping.get(v, "(unchanged)") if v != "(blank)" else "(blank - stays blank)" for v in table["raw_value"]]
    print(table.to_string(index=False))
    unused = [k for k in mapping if k not in before.index]
    assert not unused, "repair keys that never occur: " + str(unused)
    for k in mapping:
        if k.isdigit() and int(k) > 30000:
            print("   Excel serial", k, "=", datetime.date(1899, 12, 30) + datetime.timedelta(days=int(k)))
    df[col] = df[col].replace(mapping)
    left = sorted(set(df[col].dropna()) - set(order))
    assert not left, "values outside the expected levels: " + str(left)
    after = df[col].value_counts(dropna=False)
    print("after repair, in the expected order:", {k: int(after.get(k, 0)) for k in order}, "| blank:", int(df[col].isna().sum()))
    if mapping:
        repaired_cols.append(idx)
log("3.1 repairs", "damaged band labels repaired in columns " + str(repaired_cols) + "; all other single-answer columns untouched")

columns with the text artifact (cleaned like your clean_survey_col, levels = what is observed after cleaning): [94, 141, 152, 170, 180, 181, 187, 210]
--- 4 | Years IT / Programming Experience
       raw_value  n_rows           repaired_to
     2 - 5 years    8037                   2-5
       11+ years    6001                   11+
    6 - 10 years    5769                  6-10
     1 - 2 years    3373                    <2
Less than 1 year    1647                    <2
         (blank)    1259 (blank - stays blank)
after repair, in the expected order: {'<2': 5020, '2-5': 8037, '6-10': 5769, '11+': 6001} | blank: 1259
--- 105 | Compensation
          raw_value  n_rows           repaired_to
            (blank)    6605 (blank - stays blank)
  Less than $20,000    4000              <$20,000
  $20,000 - $40,000    2732           (unchanged)
     Rather not say    2628           (unchanged)
  $40,000 - $60,000    2429           (unchanged)
  $60,000 - $80,000    2007           (unchanged)
 

## Gate 4 - Rows: junk, salary

### Cell 4.1 - Junk rows: propose
**Purpose:** respondents who answered nothing except Country/State (US State is routing, not an answer). **Decision made:** none yet.

In [13]:
SALARY_IDX = [spec for kind, name, status, spec, expect, note in ITEMS if kind == "salary"][0]
SAL = RAW[SALARY_IDX]
JUNK_BASE = (RAW[0],)
other_cols = [c for c in df.columns if c not in JUNK_BASE]
print("columns that do not count as an answer (country / state routing):", [c[:40] for c in JUNK_BASE])
answered_other = df[other_cols].notna().sum(axis=1)
is_junk = answered_other == 0
print("rows the rule would remove:", int(is_junk.sum()), "| of those with a salary band:", int((is_junk & df[SAL].notna()).sum()))

columns that do not count as an answer (country / state routing): ['Country']
rows the rule would remove: 48 | of those with a salary band: 0


### Cell 4.2 - Junk rows: apply
**Purpose / decision:** drop them (none has a salary). Counts logged.

In [14]:
rows_before = len(df)
df = df.loc[~is_junk].copy()
log_rows("4.2 junk rows (only Country/State answered)", rows_before, len(df))
log("4.2 junk rows", str(rows_before - len(df)) + " rows removed")

[ROWS] 4.2 junk rows (only Country/State answered) : 26086 -> 26038 (removed 48 )
[LOG] 4.2 junk rows - 48 rows removed


### Cell 4.3 - Salary band table
**Purpose:** the answer is a text band, not a number. This table fixes the order, the bounds and the midpoint used to make an annual-USD number, and lists the two non-amount answers.
**Decision made:** `CompTotal` keeps the original band text; `ConvertedCompYearly` is the band midpoint in annual USD (the question says "including bonus"). The 2015 bands run up to `>$160,000`, and that open-ended band takes `CONFIG["top_band_midpoint"]` = 170,000 (lower bound + 10,000, the convention behind 2011-2014's 150,000 for `>$140,000`, and equal to the file's own midpoint). **The top band starts at a different point than in 2011-2014 ($140,000 then, $160,000 now), so the top-coding of `ConvertedCompYearly` differs between years.** The `Unemployed` answer was relabelled `"0"` in Gate 3 (your mapping) and is dropped like `Student / Unemployed` in earlier years. Pay before 2019 is base salary only in the shared schema notes, so the band-midpoint construction is a 2011-2015 convention.

In [15]:
STUDENT_LABEL = "0"
STUDENT_DESC = 'Unemployed (relabelled "0")'
RNS_LABEL = "Rather not say"
BANDS = [
    ("<$20,000", 0, 20000, 10000), ("$20,000 - $40,000", 20000, 40000, 30000), ("$40,000 - $60,000", 40000, 60000, 50000),
    ("$60,000 - $80,000", 60000, 80000, 70000), ("$80,000 - $100,000", 80000, 100000, 90000),
    ("$100,000 - $120,000", 100000, 120000, 110000), ("$120,000 - $140,000", 120000, 140000, 130000),
    ("$140,000 - $160,000", 140000, 160000, 150000), (">$160,000", 160000, None, CONFIG["top_band_midpoint"]),
]
observed = set(df[SAL].dropna().unique())
known = set(b[0] for b in BANDS) | {STUDENT_LABEL, RNS_LABEL}
assert observed <= known, sorted(observed - known)
counts = df[SAL].value_counts()
print(pd.DataFrame([[b[0], int(counts.get(b[0], 0)), b[1], b[2], b[3]] for b in BANDS] +
                   [[STUDENT_LABEL, int(counts.get(STUDENT_LABEL, 0)), None, None, None], [RNS_LABEL, int(counts.get(RNS_LABEL, 0)), None, None, None]],
                   columns=["answer", "n_rows", "lower", "upper", "midpoint_used"]).to_string(index=False))
print("missing salary answer:", int(df[SAL].isna().sum()))

             answer  n_rows    lower    upper  midpoint_used
           <$20,000    4000      0.0  20000.0        10000.0
  $20,000 - $40,000    2732  20000.0  40000.0        30000.0
  $40,000 - $60,000    2429  40000.0  60000.0        50000.0
  $60,000 - $80,000    2007  60000.0  80000.0        70000.0
 $80,000 - $100,000    1394  80000.0 100000.0        90000.0
$100,000 - $120,000     991 100000.0 120000.0       110000.0
$120,000 - $140,000     562 120000.0 140000.0       130000.0
$140,000 - $160,000     280 140000.0 160000.0       150000.0
          >$160,000     462 160000.0      NaN       170000.0
                  0    1996      NaN      NaN            NaN
     Rather not say    2628      NaN      NaN            NaN
missing salary answer: 6557


### Cell 4.3b - Check the band midpoints against the file's own midpoint column
**Purpose:** the 2015 file ships a `Compensation: midpoint` column. Instead of trusting either table, compare them row by row.
**Decision made:** none; the cell stops if even one row disagrees. The file's column is not carried into the output (it is listed in Cell 2.0 as redundant).

In [16]:
file_mid = pd.to_numeric(df[RAW[106]], errors="coerce")
ours = df[SAL].map({b[0]: b[3] for b in BANDS})
both = ours.notna()
print("rows with a salary band:", int(both.sum()), "| file midpoint equals the midpoint used here:", int((file_mid[both] == ours[both]).sum()))
assert (file_mid[both] == ours[both]).all(), "the file's midpoint differs from the band table"
print("what the file's midpoint column holds for the non-amount answers:", df.loc[~both & df[SAL].notna(), RAW[106]].value_counts().to_dict())
log("4.3b midpoints", "the file's own midpoint column agrees with the band table on all " + str(int(both.sum())) + " rows with a band")

rows with a salary band: 14857 | file midpoint equals the midpoint used here: 14857
what the file's midpoint column holds for the non-amount answers: {'Rather not say': 2628, 'Unemployed': 1996}
[LOG] 4.3b midpoints - the file's own midpoint column agrees with the band table on all 14857 rows with a band


### Cell 4.4 - Drop rows with no usable salary
**Purpose / decision:** drop blank salary, then (per `CONFIG`) `Rather not say` and `Student / Unemployed`: none of the three is an amount. Counts logged after each step.

In [17]:
rows_before = len(df)
df = df.loc[df[SAL].notna()].copy()
log_rows("4.4 salary answer blank", rows_before, len(df))
if CONFIG["drop_rather_not_say"]:
    rows_before = len(df)
    df = df.loc[df[SAL] != RNS_LABEL].copy()
    log_rows("4.4 'Rather not say'", rows_before, len(df))
if CONFIG["drop_student_unemployed"]:
    rows_before = len(df)
    df = df.loc[df[SAL] != STUDENT_LABEL].copy()
    log_rows("4.4 " + STUDENT_DESC, rows_before, len(df))
assert set(df[SAL].unique()) <= set(b[0] for b in BANDS)
log("4.4 salary rows", "rows without a salary amount removed; " + str(len(df)) + " remain")

[ROWS] 4.4 salary answer blank : 26038 -> 19481 (removed 6557 )
[ROWS] 4.4 'Rather not say' : 19481 -> 16853 (removed 2628 )
[ROWS] 4.4 Unemployed (relabelled "0") : 16853 -> 14857 (removed 1996 )
[LOG] 4.4 salary rows - rows without a salary amount removed; 14857 remain


## Gate 5 - Multi-answer questions stored as lists

### Cell 5.1 - Helper and all list blocks
**Purpose:** these questions have one column per option. 2016+ files store the same kind of question as one `;`-separated text cell, so every year is written that way and pooled multi-hot encoding is later one function for every year.
**Decisions made:** option labels come from row 0; `Other (please specify)` becomes `Other` (its typed text is kept in its own free-text column, not lost); every option cell must equal its label (asserted); labels are asserted to contain no `;`; a respondent who ticked nothing gets an empty cell (NaN), not an empty string. Options are listed in questionnaire order. Language blocks are handled in Gate 6.

In [18]:
def block_to_list(frame, a, b):
    cols = RAW[a:b]
    labels = [label(i) for i in range(a, b)]
    is_other = [str(l).lower().startswith("other (please specify)") for l in labels]
    nice = ["Other" if o else l for l, o in zip(labels, is_other)]
    assert all(";" not in l for l in nice), "an option label contains ';'"
    for col, l, o in zip(cols, labels, is_other):
        if not o:
            assert (frame[col].dropna() == l).all(), "cell text differs from the option label in " + col
    def join_row(values):
        picked = [n for n, v in zip(nice, values) if pd.notna(v)]
        return ";".join(picked) if picked else np.nan
    series = frame[cols].apply(lambda r: join_row(r.values), axis=1)
    return series, nice

LISTS = {}
for kind, name, status, spec, expect, note in ITEMS:
    if kind != "list":
        continue
    series, nice = block_to_list(df, spec[0], spec[1])
    LISTS[name] = series
    per_row = df[RAW[spec[0]:spec[1]]].notna().sum(axis=1)
    print("---", name, "| cols", spec[0], "to", spec[1] - 1, "| blank:", int(series.isna().sum()))
    print("   options and rows that ticked each:", {n: int(df[c].notna().sum()) for n, c in zip(nice, RAW[spec[0]:spec[1]])})
    print("   rows by number of options ticked:", per_row.value_counts().sort_index().to_dict())
    print("   example:", series.dropna().iloc[0])
log("5.1 list blocks", str(len(LISTS)) + " multi-answer blocks stored as ';' lists: " + ", ".join(LISTS))

--- TechWantToWorkWith | cols 51 to 93 | blank: 384
   options and rows that ticked each: {'Android': 3793, 'Arduino / Raspberry Pi': 1826, 'AngularJS': 3181, 'C': 1496, 'C++': 2056, 'C++11': 1604, 'C#': 4074, 'Cassandra': 306, 'CoffeeScript': 768, 'Cordova': 452, 'Clojure': 550, 'Cloud (AWS, GAE, Azure, etc.)': 1354, 'Dart': 295, 'F#': 660, 'Go': 1361, 'Hadoop': 835, 'Haskell': 830, 'iOS': 2147, 'Java': 3860, 'JavaScript': 6023, 'LAMP': 659, 'Matlab': 300, 'MongoDB': 1871, 'Node.js': 3398, 'Objective-C': 1158, 'Perl': 407, 'PHP': 2463, 'Python': 3947, 'R': 853, 'Redis': 870, 'Ruby': 1775, 'Rust': 536, 'Salesforce': 108, 'Scala': 1090, 'Sharepoint': 209, 'Spark': 308, 'SQL': 3670, 'SQL Server': 1757, 'Swift': 1709, 'Visual Basic': 397, 'Windows Phone': 736, 'Wordpress': 672, 'Other': 1242}
   rows by number of options ticked: {0: 384, 1: 1290, 2: 2038, 3: 2514, 4: 2281, 5: 1904, 6: 1412, 7: 959, 8: 660, 9: 523, 10: 309, 11: 198, 12: 138, 13: 78, 14: 57, 15: 34, 16: 18, 17: 18, 18: 8, 1

## Gate 6 - Language proficiency

### Cell 6.1 - Look at the block
**Purpose:** print the tick-box labels and the write-in volume before deciding anything.
**Decision made:** none.

In [19]:
LANG_SPEC = [spec for kind, name, status, spec, expect, note in ITEMS if kind == "lang"][0]
a, b = LANG_SPEC
lang_cols = RAW[a:b]
lang_labels = [label(i) for i in range(a, b)]
for i, lab in zip(range(a, b), lang_labels):
    print("col", i, "->", lab, "| ticked:", int(df[RAW[i]].notna().sum()))
WRITEIN_COL = lang_cols[-1]
assert lang_labels[-1].lower().startswith("other (please specify)")
writein = df[WRITEIN_COL].dropna()
print()
print("write-in answers:", len(writein), "| distinct as typed:", writein.nunique(), "| distinct lowercased:", writein.str.lower().str.strip().nunique())
print("most common (lowercased):")
print(writein.str.lower().str.strip().value_counts().head(15).to_string())

col 8 -> Android | ticked: 2654
col 9 -> Arduino / Raspberry Pi | ticked: 977
col 10 -> AngularJS | ticked: 2193
col 11 -> C | ticked: 2052
col 12 -> C++ | ticked: 2646
col 13 -> C++11 | ticked: 1069
col 14 -> C# | ticked: 4885
col 15 -> Cassandra | ticked: 156
col 16 -> CoffeeScript | ticked: 591
col 17 -> Cordova | ticked: 463
col 18 -> Clojure | ticked: 131
col 19 -> Cloud (AWS, GAE, Azure, etc.) | ticked: 1120
col 20 -> Dart | ticked: 68
col 21 -> F# | ticked: 125
col 22 -> Go | ticked: 336
col 23 -> Hadoop | ticked: 244
col 24 -> Haskell | ticked: 188
col 25 -> iOS | ticked: 1393
col 26 -> Java | ticked: 5142
col 27 -> JavaScript | ticked: 8460
col 28 -> LAMP | ticked: 1447
col 29 -> Matlab | ticked: 497
col 30 -> MongoDB | ticked: 1294
col 31 -> Node.js | ticked: 2120
col 32 -> Objective-C | ticked: 1230
col 33 -> Perl | ticked: 510
col 34 -> PHP | ticked: 4325
col 35 -> Python | ticked: 3334
col 36 -> R | ticked: 505
col 37 -> Redis | ticked: 686
col 38 -> Ruby | ticked: 1279
co

### Cell 6.2 - What the tick-boxes become (canonical names)
**Purpose:** a tick-box and a write-in must end up with the same name. This table maps each tick-box to a language, a tool, or "none".
**Decisions made (review me):**
- `CSS` and `HTML5` both become **`HTML/CSS`** (a respondent who ticked both gets it once).
- `None` means no languages: it adds nothing to the list and the count is 0 (not missing).
- **Tools are not languages.** If a tick-box is a tool (`jQuery`, `Node.js` in 2013) it goes to the separate column `ToolsHaveWorkedWith`, not into the language list or count. Both `jQuery` and `JQuery` boxes exist in 2013 and are the same option.
- Canonical names follow the 2021+ style (`Bash/Shell`, `HTML/CSS`, `Objective-C`).

In [20]:
TICK_MAP = {
    "C": ("lang", "C"), "C++": ("lang", "C++"), "C++11": ("lang", "C++"), "C#": ("lang", "C#"), "Clojure": ("lang", "Clojure"),
    "CoffeeScript": ("lang", "CoffeeScript"), "Dart": ("lang", "Dart"), "F#": ("lang", "F#"), "Go": ("lang", "Go"), "Haskell": ("lang", "Haskell"),
    "Java": ("lang", "Java"), "JavaScript": ("lang", "JavaScript"), "Matlab": ("lang", "MATLAB"), "Objective-C": ("lang", "Objective-C"),
    "Perl": ("lang", "Perl"), "PHP": ("lang", "PHP"), "Python": ("lang", "Python"), "R": ("lang", "R"), "Ruby": ("lang", "Ruby"),
    "Rust": ("lang", "Rust"), "Scala": ("lang", "Scala"), "SQL": ("lang", "SQL"), "Swift": ("lang", "Swift"), "Visual Basic": ("lang", "Visual Basic"),
    "Android": ("tool", "Android"), "Arduino / Raspberry Pi": ("tool", "Arduino / Raspberry Pi"), "AngularJS": ("tool", "AngularJS"),
    "Cassandra": ("tool", "Cassandra"), "Cordova": ("tool", "Cordova"), "Cloud (AWS, GAE, Azure, etc.)": ("tool", "Cloud"),
    "Hadoop": ("tool", "Hadoop"), "iOS": ("tool", "iOS"), "LAMP": ("tool", "LAMP"), "MongoDB": ("tool", "MongoDB"), "Node.js": ("tool", "Node.js"),
    "Redis": ("tool", "Redis"), "Salesforce": ("tool", "Salesforce"), "Sharepoint": ("tool", "SharePoint"), "Spark": ("tool", "Spark"),
    "SQL Server": ("tool", "SQL Server"), "Windows Phone": ("tool", "Windows Phone"), "Wordpress": ("tool", "WordPress"),
}
print(pd.DataFrame([[k, v[0], v[1]] for k, v in TICK_MAP.items()], columns=["tick-box label (row 0)", "goes to", "canonical name"]).to_string(index=False))
assert set(lang_labels[:-1]) == set(TICK_MAP), sorted(set(lang_labels[:-1]) ^ set(TICK_MAP))

       tick-box label (row 0) goes to         canonical name
                            C    lang                      C
                          C++    lang                    C++
                        C++11    lang                    C++
                           C#    lang                     C#
                      Clojure    lang                Clojure
                 CoffeeScript    lang           CoffeeScript
                         Dart    lang                   Dart
                           F#    lang                     F#
                           Go    lang                     Go
                      Haskell    lang                Haskell
                         Java    lang                   Java
                   JavaScript    lang             JavaScript
                       Matlab    lang                 MATLAB
                  Objective-C    lang            Objective-C
                         Perl    lang                   Perl
                        

### Cell 6.3 - Alias dictionary for the write-ins
**Purpose:** turn free text into canonical names. People typed several languages per box, in mixed case, with typos, abbreviations and commentary.
**Method:** each rule is a pattern searched anywhere in the lowercased text, and a match is *removed* before the next rule runs, so `vb.net` is not also counted as `vb`. More specific rules come first.
**This is the 2011 dictionary plus additions** needed by the 2012/2013 write-ins (CoffeeScript, TypeScript, Dart, Go, Racket, Octave, XQuery, XSLT, Apex, PL/pgSQL, CUDA, ...). A rule that is shared by all years keeps the same canonical name, so the years pool cleanly.
**Decisions made (review me):**
- **Languages are counted; tools are not.** Frameworks, libraries, databases and platforms (jQuery, Django, Rails, MongoDB, MySQL, ASP.NET, Node.js...) are recognised and stored in `ToolsHaveWorkedWith`, but kept out of the language list and count. Databases are tools: a respondent who typed `MySQL` is not counted as knowing `SQL`.
- `Ruby on Rails` counts `Ruby` (Rails implies Ruby) and records `Rails` as a tool.
- **Visual Basic stays split three ways:** `VB.NET`, `Visual Basic 6` and plain `Visual Basic`.
- One-letter names (`R`, `C`, `D`, `J`, `Go`) only count when that is the whole phrase.
- Anything left over is kept as `unrecognized` for audit and never counted. Real languages typed that this dictionary does not know are therefore **not** counted, so counts are a slight undercount for rare languages.

In [21]:
def _name_regex(name):
    return r"\b" + re.escape(name.lower()).replace(r"\ ", r"\s+") + r"\b"

LANG_RULES = [
    ("VB.NET", r"visual\s*basic\s*\.?\s*net|\bvb\s*\.?\s*net"),
    ("VBScript", r"\bvb\s*script|\bvbs\b"),
    ("VBA", r"\bvba\b|visual\s*basic\s+for\s+applications"),
    ("Visual Basic 6", r"\bvb\s*6\b|visual\s*basic\s*6|classic\s*vb"),
    ("Visual Basic", r"visual\s*basic|\bvb\b"),
    ("PL/SQL", r"\bpl\s*/\s*sql"),
    ("SQL", r"\bsql\b"),
    ("Objective-C", r"\bobj(?:ective|ectve)?[\s\-]*c\b"),
    ("C#", r"\bc\s?#"),
    ("C++", r"\bc\s?\+\+|\bcpp\b"),
    ("F#", r"\bf\s?#"),
    ("Java", r"\bjava\b"),
    ("JavaScript", r"\bjavascript\b|\becmascript\b"),
    ("PHP", r"\bphp\b"),
    ("Python", r"\bpython\b"),
    ("Ruby", r"\bruby\b"),
    ("Perl", r"\bperl\b"),
    ("HTML/CSS", r"\bx?html5?\b|\bcss3?\b"),
    ("PowerShell", r"powershell"),
    ("Bash/Shell", r"\bbash\b|\bshell?(?:\s+script\w*)?\b|\bksh\b|\bzsh\b|\bcsh\b|\bsh\b"),
    ("Assembly", r"(?:x86\s+|arm\s+|z/)?assembl(?:y|er)\b(?:\s+language)?"),
    ("Delphi", r"delphi|object\s*pascal|\boo-pascal\b"),
    ("Pascal", r"\bpascal\b"),
    ("ActionScript", r"actionscrip\w*(?:\s*3(?:\.0)?)?|\bas[123](?:-[123])?\b"),
    ("ColdFusion", r"coldfusion|\bcfml\b"),
    ("Lisp", r"\blisp\b"),
    ("MEL", r"\bmel\b"),
    # ---- additions for 2012/2013 ----
    ("PL/pgSQL", r"\bpl\s*/?\s*pgsql\b"),
    ("PL/SQL", r"\bplsql\b"),
    ("CoffeeScript", r"\bcoff?e+s+cript\b|\bcoffee\s+script\b"),
    ("TypeScript", r"\btypescript\b"),
    ("Dart", r"\bdart\b"),
    ("Go", r"\bgolang\b|\bgoogle\s+go\b"),
    ("Racket", r"\bracket\b"),
    ("Octave", r"\boctave\b"),
    ("XQuery", r"\bxquery\b"),
    ("XSLT", r"\bxslt?\b"),
    ("Apex", r"\bapex\b"),
    ("Emacs Lisp", r"\belisp\b"),
    ("PostScript", r"\bpostscript\b"),
    ("Haxe", r"\bhaxe\b"),
    ("QML", r"\bqml\b"),
    ("HLSL", r"\bhlsl\b"),
    ("CUDA", r"\bcuda\b"),
    ("SML", r"\bsml\b"),
    ("APL", r"\bapl\b"),
    ("Pawn", r"\bpawn\b"),
    ("BrightScript", r"\bbrightscript\b"),
    ("RPG", r"\brpgle\b"),
    ("FoxPro", r"visual\s*foxpro|\bvfp\b"),
    ("Progress ABL", r"progress\s+(?:openedge|4gl|abl)|\bopenedge\b"),
    ("Assembly", r"\basm\b|\bnasm\b"),
    ("ActionScript", r"\baction\s+script(?:\s*3(?:\.0)?)?\b"),
    ("ColdFusion", r"\bcold\s+fusion\b"),
    ("HTML/CSS", r"\bhtml4\b"),
    ("Pascal", r"free\s*pascal"),
    ("Julia", r"\bjulia\b"),
    ("Coq", r"\bcoq\b"),
    ("Stata", r"\bstata\b"),
    ("SystemVerilog", r"\bsystemverilog\b"),
    ("GLSL", r"\bglsl\b"),
    ("Kotlin", r"\bkotlin\b"),
    ("Elixir", r"\belix[ie]r\b"),
    ("OpenCL", r"\bopencl\b"),
    ("PureScript", r"\bpurescript\b"),
    ("LiveScript", r"\blives*cript\b"),
    ("X++", r"\bx\+\+"),
    ("Bash/Shell", r"\bbourne(?:\s+shell)?\b|\bkorn(?:\s+shell)?\b"),
]
RARE_LANGS = ["Scheme", "Clojure", "Haskell", "Scala", "Erlang", "Lua", "Groovy", "OCaml", "Prolog", "Fortran", "COBOL", "Ada",
              "Tcl", "AWK", "Smalltalk", "Eiffel", "Forth", "Rexx", "MATLAB", "Mathematica", "Maple", "IDL", "SAS", "RPG", "ABAP",
              "PowerBuilder", "FoxPro", "LabVIEW", "AppleScript", "VHDL", "Verilog", "AutoHotkey", "AutoIt", "LSL", "Avisynth",
              "KRL", "Wasabi", "GeneXus", "LotusScript", "UniBasic", "UniQuery", "Nemerle", "Gosu", "Vala", "Brainfuck", "Befunge",
              "LOLCODE", "RealBasic", "CCScript", "DataFlex", "WebFocus", "Linc 4GL", "Forte 4GL", "Progress ABL",
              "JMP Scripting Language", "XSLT", "Lansa", "BASIC"]
LANG_RULES += [(n, _name_regex(n)) for n in RARE_LANGS]

TOOL_RULES = [
    ("ASP.NET", r"\basp\.net(?:-mvc)?"), ("ASP", r"(?:classic\s+)?\basp\b(?:\s*(?:classic|3\.0))?"), (".NET", r"(?<!\w)\.net\b"),
    ("jQuery", r"\bjquery\b"), ("JSP", r"\bjsp\b"), ("Symfony", r"\bsymfony\b"), ("Silverlight", r"\bsilverlight\b"),
    ("WPF", r"\bwpf\b"), ("WCF", r"\bwcf\b"), ("Entity Framework", r"entity-framework"), ("Grails", r"\bgrails\b"),
    ("Cocoa", r"\bcocoa\b"), ("Android", r"\bandroid\b"), ("Flex", r"(?:adobe\s+)?\bflex\b"), ("Flash", r"\bflash\b"),
    ("Excel", r"\bexcel\b"), ("NumPy", r"\bnumpy\b"), ("LaTeX", r"\blatex\b"), ("XML", r"\bxml\b"), ("XAML", r"\bxaml\b"),
    ("X3D", r"\bx3d\b"), ("XPath", r"\bxpath\b"), ("sed", r"\bsed\b"),
    # ---- additions for 2012/2013 ----
    (".NET", r"\bdot\s*net\b"), ("Node.js", r"\bnode\.?js\b|\bnode\b"), ("Django", r"\bdjango\b"), ("Rails", r"\b(?:on\s+)?rails\b"),
    ("Spring", r"\bspring\b"), ("Hibernate", r"\bhibernate\b"), ("Struts", r"\bstruts2?\b"), ("MVC", r"\bmvc\b"),
    ("Dojo", r"\bdojo\b"), ("Ext JS", r"\bext\s?js\b"), ("AngularJS", r"\bangular(?:\s?js)?\b"), ("Backbone.js", r"\bbackbone(?:\.js)?\b"),
    ("Knockout.js", r"\bknockout(?:\.?js)?\b"), ("Underscore.js", r"\bunderscore(?:\.js)?\b"), ("D3.js", r"\bd3(?:\.js)?\b"),
    ("MooTools", r"\bmootools\b"), ("GWT", r"\bgwt\b"), ("JSF", r"\bjsf\b"), ("YUI", r"\byui\b"), ("Zend Framework", r"\bzend(?:\s+framework)?\b"),
    ("Play Framework", r"\bplay\s+framework(?:\s*\d)?\b"), ("Drupal", r"\bdrupal\b"), ("Magento", r"\bmagento\b"), ("Razor", r"\brazor\b"),
    ("LINQ", r"\blinq\b"), ("Qt", r"\bqt\b"), ("AJAX", r"\bajax\b"), ("JSON", r"\bjson\b"), ("SSIS/SSRS", r"\bss[ir]s\b"),
    ("Puppet", r"\bpuppet\b"), ("Git", r"\bgit\b"), ("nginx", r"\bnginx\b"), ("MongoDB", r"\bmongo(?:db)?\b"), ("MySQL", r"\bmysql\b"),
    ("PostgreSQL", r"\bpostgres(?:ql)?\b"), ("Redis", r"\bredis\b"), ("CouchDB", r"\bcouchdb\b"), ("Neo4j", r"\bneo4j\b"),
    ("Oracle", r"\boracle\b"), ("Solr", r"\bsolr\b"), ("Elasticsearch", r"\belasticsearch\b"), ("Sass/Less", r"\bsass\b|\bscss\b"),
    ("PhoneGap", r"\bphonegap\b"), ("SharePoint", r"\bsharepoint\b"), ("jQuery", r"\bjquery\b"),
    ("Hadoop", r"\bhadoop\b"), ("Spark", r"\bspark\b"), ("Cassandra", r"\bcassandra\b"), ("Gradle", r"\bgradle\b"),
    ("React", r"\breact(?:\.?js)?\b"), ("Ember.js", r"\bember(?:\.?js)?\b"), ("Joomla", r"\bjoomla\b"), ("Meteor", r"\bmeteor(?:\.?js)?\b"),
    ("Laravel", r"\blaravel\b"), ("Xamarin", r"\bxamarin\b"), ("Docker", r"\bdocker\b"), ("Unity", r"\bunity(?:\s*3d)?\b"),
    ("Bootstrap", r"\b(?:twitter\s+)?bootstrap\b"), ("RabbitMQ", r"\brabbitmq\b"), ("RavenDB", r"\bravendb\b"), ("BizTalk", r"\bbiztalk\b"),
    ("Flask", r"\bflask\b"), ("CakePHP", r"\bcake\s*php\b"), ("CodeIgniter", r"\bcode\s*igniter\b"), ("Sitecore", r"\bsitecore\b"),
    ("Polymer", r"\bpolymer\b"), ("OpenGL", r"\b(?:open|web)gl\b"), ("DB2", r"\bdb2\b"), ("Symfony", r"\bsymfony2\b"), ("Grunt", r"\bgrunt(?:\s*js)?\b"),
]
EXACT_LANG = {"r": "R", "d": "D", "j": "J", "go": "Go", "c": "C"}
EXACT_TOOL = {"make": "Make", "less": "Sass/Less"}
# two answers where commentary contains language-like words; decided by hand and shown in Cell 5.5
MANUAL = {
    "x++ (no, that is not a misspelled c++)": {"X++"},
    "ravendb (similar to mongodb)": set(),
"css is not a programming language.": set(),
          "vb (why the fuck was this not a choice? c# snobs?)": {"Visual Basic"}}

COMPILED_LANG = [(n, re.compile(p)) for n, p in LANG_RULES]
COMPILED_TOOL = [(n, re.compile(p)) for n, p in TOOL_RULES]

def parse_writein(text):
    s = str(text).lower().strip().replace("’", "'")
    if s in MANUAL:
        return set(MANUAL[s]), set(), ["(manual override)"]
    langs, tools = set(), set()
    for name, rx in COMPILED_LANG:
        if rx.search(s):
            langs.add(name)
            s = rx.sub(" | ", s)
    for name, rx in COMPILED_TOOL:
        if rx.search(s):
            tools.add(name)
            s = rx.sub(" | ", s)
    leftovers = []
    for phrase in re.split(r"[|,;/&+()\[\]:!?]+", s):
        p = re.sub(r"[^a-z0-9#+.\- ]", "", phrase).strip(" .-")
        if not p:
            continue
        if p in EXACT_LANG:
            langs.add(EXACT_LANG[p])
        elif p in EXACT_TOOL:
            tools.add(EXACT_TOOL[p])
        else:
            leftovers.append(p)
    return langs, tools, leftovers

alias_table = pd.DataFrame([[n, "language", p] for n, p in LANG_RULES] + [[n, "tool/markup (not counted)", p] for n, p in TOOL_RULES] +
                           [[v, "language (exact phrase only)", "^" + k + "$"] for k, v in EXACT_LANG.items()],
                           columns=["canonical_name", "kind", "pattern"])
print("alias rules:", len(alias_table), "| language names:", int((alias_table.kind == "language").sum()), "| tool names:", int(alias_table.kind.str.startswith("tool").sum()))
for test in ["vb.net, vb6", "coffeescript, golang", "ruby on rails, mysql", "node.js and c", "x86 asm", "plsql"]:
    print(repr(test), "->", parse_writein(test))

alias rules: 224 | language names: 126 | tool names: 93
'vb.net, vb6' -> ({'Visual Basic 6', 'VB.NET'}, set(), [])
'coffeescript, golang' -> ({'Go', 'CoffeeScript'}, set(), [])
'ruby on rails, mysql' -> ({'Ruby'}, {'MySQL', 'Rails'}, [])
'node.js and c' -> (set(), {'Node.js'}, ['and c'])
'x86 asm' -> ({'Assembly'}, set(), ['x86'])
'plsql' -> ({'PL/SQL'}, set(), [])


### Cell 6.4 - Parse every write-in and show the result
**Purpose:** apply the dictionary to all write-ins in the retained rows. The table shows, for the canonical names found, how many respondents wrote each one, and the tools recognised.
**Decision made:** none; this is the audit step.

In [22]:
parsed = {}
for idx, text in writein.items():
    parsed[idx] = parse_writein(text)

lang_counts, tool_counts = {}, {}
for idx, (langs, tools, left) in parsed.items():
    for name in langs:
        lang_counts[name] = lang_counts.get(name, 0) + 1
    for name in tools:
        tool_counts[name] = tool_counts.get(name, 0) + 1
print("languages recognised in write-ins (respondents):")
print(pd.Series(lang_counts).sort_values(ascending=False).to_frame("respondents").T.to_string())
print()
print("tools recognised in write-ins (not counted as languages):")
print(pd.Series(tool_counts).sort_values(ascending=False).to_frame("respondents").T.to_string())
print()
n_none = sum(1 for v in parsed.values() if not v[0])
print("write-ins with no language recognised:", n_none, "of", len(parsed), "| of those with at least one tool:", sum(1 for v in parsed.values() if not v[0] and v[1]))

languages recognised in write-ins (respondents):
             Bash/Shell  Groovy  Delphi  HTML/CSS  TypeScript  PowerShell  Lua  ActionScript  Erlang  ColdFusion  Fortran  Assembly  PL/SQL  Lisp  VBA  Tcl  X++  VHDL  Verilog  XSLT  Julia  SAS  OCaml  Scheme  ABAP  D  Pascal  CUDA  Haxe  QML  COBOL  Prolog  Smalltalk  Mathematica  LabVIEW  C++  VBScript  Ada  RPG  PowerBuilder  OpenCL  Elixir  XQuery  Apex  SQL  AWK  Kotlin  Racket  LiveScript  JavaScript  Progress ABL  SystemVerilog  PureScript  FoxPro  GLSL  Ruby  VB.NET  Java  SML  MATLAB  Octave  C  IDL  Vala  PHP  PL/pgSQL  Stata  AutoIt  BrightScript  Visual Basic  APL  Visual Basic 6  UniBasic  Perl  Gosu  LotusScript  Go  CoffeeScript  BASIC  AppleScript  AutoHotkey  Python  Coq  J  Emacs Lisp  PostScript
respondents          97      95      82        78          65          45   39            37      35          34       30        28      26    20   16   16   13    13       12    11     11   10     10      10     9  9       9  

### Cell 6.5 - Review the odd ones
**Purpose:** two checks that the parser did not invent languages. (1) every write-in longer than 25 characters or with punctuation that signals commentary, with its parse (first 60 shown); (2) the most common leftover phrases, in case a real language was missed.
**Decision made:** none. Any miss is fixed by adding a rule in Cell 6.3, not by editing data.

In [23]:
long_ones = [(idx, t) for idx, t in writein.items() if len(t) > 25 or "?" in t or "!" in t]
print("commentary-style write-ins:", len(long_ones), "(first 60 shown)")
for idx, t in long_ones[:60]:
    langs, tools, left = parsed[idx]
    print("row", idx, "|", repr(t[:100]))
    print("      languages:", sorted(langs), "| tools:", sorted(tools), "| leftover:", left[:4])
print()
leftover_counts = {}
for langs, tools, left in parsed.values():
    for p in left:
        leftover_counts[p] = leftover_counts.get(p, 0) + 1
print("most common leftover phrases:")
print(pd.Series(leftover_counts).sort_values(ascending=False).head(40).to_string())

commentary-style write-ins: 76 (first 60 shown)
row 951 | 'Coldfusion, Linux shell scripting'
      languages: ['Bash/Shell', 'ColdFusion'] | tools: [] | leftover: ['linux']
row 1095 | 'Sitecore, powershell scripting'
      languages: ['PowerShell'] | tools: ['Sitecore'] | leftover: ['scripting']
row 1265 | 'iOS and Android development mostly in JS'
      languages: [] | tools: ['Android'] | leftover: ['ios and', 'development mostly in js']
row 1440 | 'Rails, NoSQL, Amazon Web Services'
      languages: [] | tools: ['Rails'] | leftover: ['nosql', 'amazon web services']
row 2383 | 'PostgreSQL, DB2, Greenplum'
      languages: [] | tools: ['DB2', 'PostgreSQL'] | leftover: ['greenplum']
row 2586 | 'Shell script is always there, and I miss Lisp from your list'
      languages: ['Bash/Shell', 'Lisp'] | tools: [] | leftover: ['is always there', 'and i miss', 'from your list']
row 2841 | 'ActionScript, Flash, Scaleform'
      languages: ['ActionScript'] | tools: ['Flash'] | leftover: ['scalef

### Cell 6.6 - Build `LanguageHaveWorkedWith`, the count, and `ToolsHaveWorkedWith`
**Purpose:** combine the tick-boxes and the parsed write-ins into one canonical list per respondent, plus a count.
**Decisions made:**
- The language list is the **distinct** canonical names, sorted alphabetically, joined with `;`.
- `LanguageHaveWorkedWith_count` is the length of that list. It is `0` only for people who ticked `None` with no language. It is empty (NaN, meaning unknown, not zero) for people who left the block blank, for people who ticked only tools (the question is "languages or technologies", so a tools-only answer does not prove they use no language), and for people whose only answer was a write-in with no recognisable language (for example "various"). Someone who ticked `None` and also named a language keeps the language.
- A name in both the tick-boxes and the write-in is counted once.
- `ToolsHaveWorkedWith` is the same kind of `;` list for tools. An empty cell there only means no tool was recognised; it does not mean unknown. In 2012 there are no tool tick-boxes, so the column comes from write-ins only.
- Caution for the pooled model: the number of tick-boxes differs every year (2011: 12, 2012: 14, 2013: 13, 2021+: 40+), so a count of 4 does not mean the same in every year. Compare counts within a year, or let the model see `survey_year`.

In [24]:
tick_cols = lang_cols[:-1]
tick_labels = lang_labels[:-1]
final_lists, final_counts, final_tools, none_flags = [], [], [], []
for idx in df.index:
    names, tools = set(), set()
    none_ticked = False
    for col, lab in zip(tick_cols, tick_labels):
        if pd.notna(df.at[idx, col]):
            kind, canon = TICK_MAP[lab]
            if kind == "none":
                none_ticked = True
            elif kind == "lang":
                names.add(canon)
            else:
                tools.add(canon)
    if idx in parsed:
        names |= parsed[idx][0]
        tools |= parsed[idx][1]
    final_lists.append(";".join(sorted(names, key=str.lower)) if names else np.nan)
    final_tools.append(";".join(sorted(tools, key=str.lower)) if tools else np.nan)
    if names:
        final_counts.append(len(names))
    elif none_ticked:
        final_counts.append(0)
    else:
        final_counts.append(np.nan)
    none_flags.append(none_ticked)

language_list = pd.Series(final_lists, index=df.index, dtype=object)
language_count = pd.Series(final_counts, index=df.index, dtype=float)
tools_list = pd.Series(final_tools, index=df.index, dtype=object)
none_ticked = pd.Series(none_flags, index=df.index)
block_answered = df[lang_cols].notna().any(axis=1)
print("rows:", len(df), "| count blank:", int(language_count.isna().sum()),
      "(block unanswered:", int((~block_answered).sum()), "| answered, no language found:", int((block_answered & language_list.isna() & ~none_ticked).sum()),
      "of which tools only:", int((block_answered & language_list.isna() & ~none_ticked & tools_list.notna()).sum()), ")")
print("'None' ticked:", int(none_ticked.sum()), "| of those, rows that also have a language:", int((none_ticked & language_list.notna()).sum()), "| count = 0 rows:", int((language_count == 0).sum()))
print("count distribution:")
print(language_count.value_counts().sort_index().to_string())
print("mean count: %.2f" % language_count.mean())
print("rows with at least one tool:", int(tools_list.notna().sum()))
log("6.6 languages", "tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list, count and ToolsHaveWorkedWith")

rows: 14857 | count blank: 250 (block unanswered: 107 | answered, no language found: 143 of which tools only: 130 )
'None' ticked: 0 | of those, rows that also have a language: 0 | count = 0 rows: 0
count distribution:
1.0     2013
2.0     3565
3.0     4050
4.0     2465
5.0     1234
6.0      670
7.0      317
8.0      136
9.0       72
10.0      38
11.0      21
12.0       5
13.0       6
14.0       7
22.0       1
23.0       7
mean count: 3.17
rows with at least one tool: 9852
[LOG] 6.6 languages - tick-boxes + parsed write-ins -> LanguageHaveWorkedWith list, count and ToolsHaveWorkedWith


### Cell 6.7 - Sanity checks and audit file
**Purpose:** confirm no name is spelled two ways, the list round-trips to the count, and write out an audit file so every parsed write-in can be checked by eye.
**Decision made:** none.

In [25]:
all_names = sorted({n for s in language_list.dropna() for n in s.split(";")}, key=str.lower)
print("distinct canonical language names in the column:", len(all_names))
print(all_names)
lowered = {}
for n in all_names:
    lowered.setdefault(n.lower(), []).append(n)
assert all(len(v) == 1 for v in lowered.values()), "two spellings of the same name"
tool_names = sorted({n for s in tools_list.dropna() for n in s.split(";")}, key=str.lower)
print("distinct tool names:", len(tool_names))
print(tool_names)
overlap = set(all_names) & set(tool_names)
assert not overlap, "a name is both a language and a tool: " + str(overlap)
for idx in df.index:
    s = language_list[idx]
    n = 0 if pd.isna(s) else len(s.split(";"))
    c = language_count[idx]
    assert (pd.isna(c) and pd.isna(s)) or c == n, "count does not match list at row " + str(idx)
print("count matches list length for every row")

audit = pd.DataFrame({"source_row": list(parsed.keys()), "as_typed": [writein[i] for i in parsed],
                      "languages_found": [";".join(sorted(parsed[i][0])) for i in parsed],
                      "tools_not_counted": [";".join(sorted(parsed[i][1])) for i in parsed],
                      "unrecognized_text": [" | ".join(parsed[i][2]) for i in parsed]})
audit.to_csv("lang_writein_audit_2015.csv", index=False)
alias_table.to_csv("language_alias_table_v2.csv", index=False)
print("saved lang_writein_audit_2015.csv (%d rows) and language_alias_table_v2.csv (%d rules)" % (len(audit), len(alias_table)))

distinct canonical language names in the column: 96
['ABAP', 'ActionScript', 'Ada', 'Apex', 'APL', 'AppleScript', 'Assembly', 'AutoHotkey', 'AutoIt', 'AWK', 'Bash/Shell', 'BASIC', 'BrightScript', 'C', 'C#', 'C++', 'Clojure', 'COBOL', 'CoffeeScript', 'ColdFusion', 'Coq', 'CUDA', 'D', 'Dart', 'Delphi', 'Elixir', 'Emacs Lisp', 'Erlang', 'F#', 'Fortran', 'FoxPro', 'GLSL', 'Go', 'Gosu', 'Groovy', 'Haskell', 'Haxe', 'HTML/CSS', 'IDL', 'J', 'Java', 'JavaScript', 'Julia', 'Kotlin', 'LabVIEW', 'Lisp', 'LiveScript', 'LotusScript', 'Lua', 'Mathematica', 'MATLAB', 'Objective-C', 'OCaml', 'Octave', 'OpenCL', 'Pascal', 'Perl', 'PHP', 'PL/pgSQL', 'PL/SQL', 'PostScript', 'PowerBuilder', 'PowerShell', 'Progress ABL', 'Prolog', 'PureScript', 'Python', 'QML', 'R', 'Racket', 'RPG', 'Ruby', 'Rust', 'SAS', 'Scala', 'Scheme', 'Smalltalk', 'SML', 'SQL', 'Stata', 'Swift', 'SystemVerilog', 'Tcl', 'TypeScript', 'UniBasic', 'Vala', 'VB.NET', 'VBA', 'VBScript', 'Verilog', 'VHDL', 'Visual Basic', 'Visual Basic 6', 

## Gate 7 - Headers and assembly

### Cell 7.1 - Header map
**Purpose:** rename every column to the shared name, and record the original header, kind, status and a note.
**Decisions made (review me):** the **confirmed** names are the ones in your shared list (`Country`, `Age`, `YearsCode`, `OrgSize`, `DevType`, `LanguageHaveWorkedWith`, `OpSys`, `JobSat`, `CompTotal`, `ConvertedCompYearly`). I could not see your Canonical Mapping spreadsheet, so every other name is **proposed**: check it against the sheet and change it in `ITEMS` (Cell 2.0). `ResponseId` is the row number in the 2015 file, since the file has no respondent id. The `kind` column says how each column is stored, so you can decide what to drop before modelling (`free_text` columns are raw typed text).

In [26]:
FIXED_IDX = set(FIXES) | set()
rows_hm = []
def span_text(spec):
    first = spec[0] if isinstance(spec, tuple) else spec
    s = full_header(first)
    if isinstance(spec, tuple):
        s = RAW[PARENT[first]] + " (cols " + str(spec[0]) + "-" + str(spec[1] - 1) + ")"
    return s
for kind, name, status, spec, expect, note in ITEMS:
    if kind == "id":
        rows_hm.append((name, status, "id", "(none in file)", note))
    elif kind == "year":
        rows_hm.append((name, status, "constant", "(added)", note))
    elif kind == "single":
        k = "single (repaired)" if spec in FIXES and FIXES[spec][0] else "single"
        rows_hm.append((name, status, k, span_text(spec), note))
    elif kind == "list":
        rows_hm.append((name, status, "list ';'", span_text(spec), note))
    elif kind == "freetext":
        rows_hm.append((name, status, "free_text", span_text(spec), note))
    elif kind == "lang":
        rows_hm.append((name, status, "list ';' (cleaned)", span_text(spec), note))
        rows_hm.append((name + "_count", "proposed", "derived number", "(derived)", "number of distinct languages; 0 if 'None' ticked; blank if block unanswered or only an unusable write-in"))
        rows_hm.append(("ToolsHaveWorkedWith", "proposed", "list ';' (cleaned)", "(derived from tick-boxes and write-ins)", "frameworks/libraries/databases/platforms recognised in the language question; not counted as languages; empty = none recognised"))
    elif kind == "salary":
        rows_hm.append((name, status, "single (band text)", span_text(spec), note))
        rows_hm.append(("ConvertedCompYearly", "confirmed", "derived number", "(derived from the salary band)", "annual USD band midpoint; includes bonus; top band set by CONFIG"))
header_map = pd.DataFrame(rows_hm, columns=["canonical_name", "status", "kind", "original_header_or_span", "note"])
print(header_map.status.value_counts().to_dict())
print(header_map[["canonical_name", "status", "kind"]].to_string(index=False))

{'proposed': 50, 'confirmed': 9}
                      canonical_name    status               kind
                          ResponseId  proposed                 id
                         survey_year  proposed           constant
                             Country confirmed             single
                                 Age confirmed             single
                              Gender  proposed             single
                        TabsOrSpaces  proposed             single
                           YearsCode confirmed  single (repaired)
                             DevType confirmed             single
                               OpSys confirmed             single
                       OpSys_WriteIn  proposed          free_text
              LanguageHaveWorkedWith confirmed list ';' (cleaned)
        LanguageHaveWorkedWith_count  proposed     derived number
                 ToolsHaveWorkedWith  proposed list ';' (cleaned)
                  TechWantToWorkWith  propo

### Cell 7.2 - Assemble the canonical table
**Purpose:** put the columns together in the original questionnaire order, with the shared names.
**Decision made:** none new. Blanks stay blank; nothing is imputed, merged or encoded.

In [27]:
label_to_mid = {bnd[0]: bnd[3] for bnd in BANDS}
out = pd.DataFrame(index=df.index)
for kind, name, status, spec, expect, note in ITEMS:
    if kind == "id":
        out[name] = df.index.astype(int)
    elif kind == "year":
        out[name] = 2015
    elif kind in ("single", "freetext"):
        out[name] = df[RAW[spec]]
    elif kind == "list":
        out[name] = LISTS[name]
    elif kind == "lang":
        out[name] = language_list
        out[name + "_count"] = language_count
        out["ToolsHaveWorkedWith"] = tools_list
    elif kind == "salary":
        out[name] = df[RAW[spec]]
        out["ConvertedCompYearly"] = df[RAW[spec]].map(label_to_mid).astype(float)

assert list(out.columns) == list(header_map.canonical_name), "header map and table columns differ"
print("shape:", out.shape)
print(out.head(3).T.to_string(max_colwidth=60))

shape: (14857, 59)
                                                                                                1                                                            2                                                            3
ResponseId                                                                                      1                                                            2                                                            3
survey_year                                                                                  2015                                                         2015                                                         2015
Country                                                                                   Croatia                                                       France                                                        India
Age                                                                                         25-29    

### Cell 7.3 - Validate
**Purpose:** prove the file is what it claims: right row count, no encoded columns, no empty strings, list columns well-formed, every single-answer column equal to the file (or to the file after the Gate 3 repair, for the repaired ones), missing values left as missing.
**Decision made:** none.

In [28]:
assert len(out) == len(df)
assert out["ResponseId"].is_unique
list_cols = [h for h, k in zip(header_map.canonical_name, header_map.kind) if k.startswith("list")]
for c in list_cols:
    s = out[c].dropna()
    assert not s.str.startswith(";").any() and not s.str.endswith(";").any() and not s.str.contains(";;").any(), c
    assert all(len(v.split(";")) == len(set(v.split(";"))) for v in s), "duplicate item in " + c
    print("%-28s | blank: %5d | mean items: %.2f" % (c, int(out[c].isna().sum()), s.str.split(";").str.len().mean()))
text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
for c in text_cols:
    assert not (out[c].dropna().str.strip() == "").any(), "empty string in " + c
numeric_cols = [c for c in out.columns if c not in text_cols]
na_like = {"None", "N/A", "NA", "n/a", "null", "NULL", "NaN", "nan"}
hits = {c: int(out[c].isin(na_like).sum()) for c in text_cols}
print("cells holding NA-like TEXT that is a real answer (read the CSV with keep_default_na=False):", {k: v for k, v in hits.items() if v})
print("numeric columns:", numeric_cols)

n_checked = 0
for kind, name, status, spec, expect, note in ITEMS:
    if kind in ("single", "freetext", "salary"):
        assert out[name].equals(df[RAW[spec]]), name + " differs from the repaired file column"
        if not (spec in FIXES and FIXES[spec][0]):
            assert out[name].equals(df_raw.loc[out.index, RAW[spec]]), name + " differs from the original file"
        n_checked += 1
print("single-answer and free-text columns checked against the file:", n_checked, "| repaired columns (compared with the repaired text):", sorted(repaired_cols))
print("missing per column:")
print(out.isna().sum()[out.isna().sum() > 0].to_string())
print("any one-hot style 0/1 columns:", [c for c in numeric_cols if set(out[c].dropna().unique()) <= {0, 1} and c != "survey_year"])

LanguageHaveWorkedWith       | blank:   250 | mean items: 3.17
ToolsHaveWorkedWith          | blank:  5005 | mean items: 2.10
TechWantToWorkWith           | blank:   384 | mean items: 4.67
TrainingAndEducation         | blank:    28 | mean items: 1.93
JobAspectsMostImportant      | blank:  1331 | mean items: 2.95
JobSearchAnnoyances          | blank:  2998 | mean items: 2.28
RecruitEmailAppealingTraits  | blank:  2968 | mean items: 2.75
JobInfoMostUrgent            | blank:  2926 | mean items: 2.99
InterviewImprovements        | blank:  3271 | mean items: 2.62
WhyTryCareers                | blank:  7412 | mean items: 2.02
SourceControlUsed            | blank:  2861 | mean items: 1.56
WhyUseStackOverflow          | blank:  2928 | mean items: 3.13
WhyAnswer                    | blank:  3110 | mean items: 2.52
cells holding NA-like TEXT that is a real answer (read the CSV with keep_default_na=False): {'Country': 175, 'TechWantToWorkWith_WriteIn': 1, 'HobbyProgrammingHours': 1042, 'Preferr

### Cell 7.4 - Save
**Purpose:** write `canonical_2015.csv` and `header_map_2015.csv`, then read the CSV back and compare.
**Decision made:** none.

In [29]:
out.to_csv("canonical_2015.csv", index=False)
header_map.to_csv("header_map_2015.csv", index=False)
back = pd.read_csv("canonical_2015.csv", keep_default_na=False, na_values=[""])
print("saved canonical_2015.csv | on disk:", back.shape, "| in memory:", out.shape)
assert back.shape == out.shape
assert list(back.columns) == list(out.columns)
assert (back.isna().values == out.isna().values).all(), "missing-value pattern changed on save"
for c in out.columns:
    if not pd.api.types.is_numeric_dtype(out[c]):
        assert (back[c].fillna("").values == out[c].fillna("").values).all(), "text changed on save in " + c
    else:
        assert np.allclose(back[c].fillna(-1).values, out[c].fillna(-1).values), "numbers changed on save in " + c
print("round trip identical: same shape, same missing pattern, same text and numbers in all", out.shape[1], "columns")
log("7.4 save", "canonical_2015.csv: " + str(out.shape[0]) + " rows x " + str(out.shape[1]) + " columns, not encoded")
print()
print("row log:")
for step, bf, af in row_log:
    print("  %-48s %5d -> %5d" % (step, bf, af))
print()
print("decision log:")
for step, text in decision_log:
    print(" ", step, "|", text)

saved canonical_2015.csv | on disk: (14857, 59) | in memory: (14857, 59)
round trip identical: same shape, same missing pattern, same text and numbers in all 59 columns
[LOG] 7.4 save - canonical_2015.csv: 14857 rows x 59 columns, not encoded

row log:
  1.2 two header rows read; no option-label row in the data 26086 -> 26086
  4.2 junk rows (only Country/State answered)      26086 -> 26038
  4.4 salary answer blank                          26038 -> 19481
  4.4 'Rather not say'                             19481 -> 16853
  4.4 Unemployed (relabelled "0")                  16853 -> 14857

decision log:
  1.1 encoding | file read with encoding=utf-8 (115556 non-ASCII byte(s) in the file)
  1.2 load | loaded 26086 rows x 222 columns, all as text, header=1, default NA parsing off
  1.4 strip | 0 column name(s) had surrounding whitespace and were stripped
  1.5 blanks | 4507861 empty/whitespace cells converted to NaN; 2199 literal 'None'-style cells kept as text
  1.6 option labels | no optio

## Written decision log (numbers come from this run)

The next cell prints the summary of what was kept, changed and dropped, built from the variables of this run, so the numbers cannot drift from the data.

In [30]:
print("OUTPUT: canonical_2015.csv,", out.shape[0], "rows x", out.shape[1], "columns; nothing one-hot encoded, imputed, merged or scaled.")
print("ROWS:   " + " -> ".join(str(r[2]) for r in row_log[:1]) + " | " + " | ".join("%s: %d -> %d" % r for r in row_log[1:]))
print("KEPT EXACTLY AS TYPED:", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k in ("single", "single (band text)")])
print("REPAIRED (Excel damage / label fixes):", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k == "single (repaired)"])
print("LISTS (';'-separated):", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k.startswith("list")])
print("FREE TEXT (raw, drop before modelling):", [n for n, k in zip(header_map.canonical_name, header_map.kind) if k == "free_text"])
print("LANGUAGES:", len(all_names), "canonical names;", len(parsed), "write-ins parsed;", len(tool_names), "tool names; mean count %.2f" % language_count.mean())
print("HEADERS: ", header_map.status.value_counts().to_dict(), "(proposed names need checking against your Canonical Mapping sheet)")
print("LOAD WITH: pd.read_csv(path, keep_default_na=False, na_values=[''])  -> real answers such as 'None' survive:", {k: v for k, v in hits.items() if v})
print("LEFT FOR THE POOLED PIPELINE (fit on training years only): one-hot / multi-hot, rare-level merging, imputation, scaling, feature selection")

OUTPUT: canonical_2015.csv, 14857 rows x 59 columns; nothing one-hot encoded, imputed, merged or scaled.
ROWS:   26086 | 4.2 junk rows (only Country/State answered): 26086 -> 26038 | 4.4 salary answer blank: 26038 -> 19481 | 4.4 'Rather not say': 19481 -> 16853 | 4.4 Unemployed (relabelled "0"): 16853 -> 14857
KEPT EXACTLY AS TYPED: ['Country', 'Age', 'Gender', 'TabsOrSpaces', 'DevType', 'OpSys', 'CompTotal', 'EmploymentStatus', 'Industry', 'JobSat', 'PurchasingPower', 'RemoteWork', 'ChangedJobsLast12m', 'OpenToNewJob', 'RemoteImportance', 'RecruiterContactFrequency', 'ContactPref_Email', 'ContactPref_LinkedIn', 'ContactPref_Xing', 'ContactPref_Phone', 'ContactPref_CareersMessage', 'ContactPref_Twitter', 'ContactPref_Facebook', 'CommunicateWith_Headhunter', 'CommunicateWith_InHouseRecruiter', 'CommunicateWith_InHouseTechRecruiter', 'CommunicateWith_Manager', 'CommunicateWith_Developer', 'CaffeinatedBeveragesPerDay', 'HobbyProgrammingHours', 'PreferredEditor', 'IDETheme', 'PreferredSour